## Imports

In [0]:
from pyspark.sql import functions as F

import uuid
import time
import json
from faker import Faker

fake = Faker()

In [0]:
dbutils.widgets.dropdown("env", "dev", ["prod", "dev", "test"], "Environment")
ENV = dbutils.widgets.get("env")

In [0]:
CATALOG = "lakehouse_dev"
SCHEMA = "cloudwatch_metrics_pull"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{ENV}"
USER_LANDING = f"{BASE_PATH}/users_json"
SPEND_LANDING = f"{BASE_PATH}/spend_csv"
csv_schema = "id,age,annual_income,spending_score"
print(BASE_PATH)

# List of directories to clean
directories = [
    f"{BASE_PATH}/users_json",
    f"{BASE_PATH}/spend_csv"
]


/Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev


In [0]:
def generate_coordinated_events(batch_size=5):
    user_events = []
    spend_events = []
    shared_ids = [fake.random_int(min=1, max=1000) for _ in range(batch_size)]
    spend_events.append(csv_schema)

    for user_id in shared_ids:
        user_events.append({
            # "id": user_id,
            "email": fake.email(),
            "creation_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"),
            "last_activity_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"),
            # "creation_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"), # break expectation 'valid_activity_sequence'
            # "last_activity_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"), 
            "firstname": fake.first_name(),
            "lastname": fake.last_name(),
            "address": fake.address().replace("\n", ", "),
            "city": fake.city(),
            "last_ip": fake.ipv4(),
            "postcode": fake.postcode()
        })
        spend_events.append(
            f"{user_id},{fake.random_int(18, 90)},{round(fake.pyfloat(left_digits=3, right_digits=1, positive=True), 1)},{round(fake.pyfloat(left_digits=2, right_digits=1, positive=True), 1)}"
        )
    return user_events, spend_events


def run_simulation(batch_size=10):
    user_data, spend_data = generate_coordinated_events(batch_size)
    unique_id = uuid.uuid4().hex[:8]
    timestamp = int(time.time())

    # Write User JSON (Newline Delimited)
    user_json_str = "\n".join([json.dumps(e) for e in user_data])
    user_file = f"{USER_LANDING}/sim_users_{timestamp}_{unique_id}.json"
    dbutils.fs.put(user_file, user_json_str, overwrite=True)

    # Write Spend CSV (No header, matching the raw ingestion pattern)
    spend_csv_str = "\n".join(spend_data)
    spend_file = f"{SPEND_LANDING}/sim_spend_{timestamp}_{unique_id}.csv"
    dbutils.fs.put(spend_file, spend_csv_str, overwrite=True)

    print(f"Simulation Aligned with Test Data:")
    print(f" - Produced {batch_size} coordinated records.")
    print(f" - Users path: {user_file}")
    print(f" - Spend path: {spend_file}")




## ⚠️ Cleanup PROD LANDING ZONE ⚠️

In [0]:
def cleanup_landing_zone():
    for directory in directories:
        try:
            # recurse=True ensures all files and subdirectories are removed
            dbutils.fs.rm(directory, recurse=True)
            # Recreate the directory so the simulation script doesn't fail on missing paths
            dbutils.fs.mkdirs(directory)
            print(f"✅ Cleaned and recreated: {directory}")
        except Exception as e:
            print(f"⚠️ Error cleaning {directory}: {e}")

In [0]:
cleanup_landing_zone()

✅ Cleaned and recreated: /Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/users_json
✅ Cleaned and recreated: /Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/spend_csv


## ⚠️ Clean up all tables ⚠️

In [0]:
def drop_medallion_tables():
    tables = [
        "user_bronze_sdp",
        "raw_user_data",
        "raw_spend_data",
        "user_silver_sdp",
        "spend_silver_sdp",
        "user_gold_sdp"
    ]

    print(f"Starting cleanup for schema: {CATALOG}.{SCHEMA}")

    for table in tables:
        table_path = f"{CATALOG}.{SCHEMA}.{table}"
        try:
            # Use IF EXISTS to prevent errors if a table hasn't been created yet
            spark.sql(f"DROP TABLE IF EXISTS {table_path}")
            print(f"✅ Successfully dropped table: {table_path}")
        except Exception as e:
            print(f"⚠️ Failed to drop {table_path}: {str(e)}")


# drop_medallion_tables()

## Produce new data

In [0]:
run_simulation(1)

Wrote 303 bytes.
Wrote 53 bytes.
Simulation Aligned with Test Data:
 - Produced 1 coordinated records.
 - Users path: /Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/users_json/sim_users_1774379332_ebd4da61.json
 - Spend path: /Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/spend_csv/sim_spend_1774379332_ebd4da61.csv


## Check Landing Zone for new files

In [0]:
print("Checking Landing Zone Volumes...")
user_files = dbutils.fs.ls(f"{USER_LANDING}")
spend_files = dbutils.fs.ls(f"{SPEND_LANDING}")

print(f"Found {len(user_files)} user files and {len(spend_files)} spend files.")

# 2. Check Processing Status via Table Metadata
# This shows you the 'last_modified' time and row counts
tables = ["user_bronze_sdp", "spend_silver_sdp", "user_gold_sdp"]

for table in tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        df = spark.table(full_name)
        count = df.count()
        # Get the latest record based on your simulated timestamps
        latest = df.select(F.max("_metadata.file_modification_time")).collect()[0][0] if "_metadata" in df.columns else "N/A"
        print(f"Table: {table} | Row Count: {count} | Latest File Processed: {latest}")
    except Exception as e:
        print(f"Could not read {full_name}: {e}")

# 3. Validation: Match User IDs between Bronze and Gold
# This confirms the join in 03-gold.py worked for your Faker data
bronze_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_bronze_sdp").select("id").distinct()
gold_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_gold_sdp").select("id").distinct()

match_count = gold_ids.intersect(bronze_ids).count()
print(f"Successful Joins in Gold Layer: {match_count}")

Checking Landing Zone Volumes...
Found 3 user files and 3 spend files.
Table: user_bronze_sdp | Row Count: 15 | Latest File Processed: N/A
Table: spend_silver_sdp | Row Count: 15 | Latest File Processed: N/A
Table: user_gold_sdp | Row Count: 0 | Latest File Processed: N/A
Successful Joins in Gold Layer: 0


## ALL

In [0]:
%sql
SELECT
  timestamp,
  -- Use dot notation to reach into the struct and array
  error.exceptions[0].message AS error_detail,

  -- Extract the specific rule name from the message string
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,

  -- Extract the input data snippet
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,

  origin.pipeline_id
FROM event_log("87915828-1d04-4162-b111-35df93ee5e15")
WHERE level = 'ERROR'
  -- Use the size function to ensure an exception actually exists before checking the message
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message IS NOT NULL
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

timestamp,error_detail,violated_rule,offending_record,pipeline_id
2026-03-24T10:35:51.826Z,"[EXPECTATION_VIOLATION.VERBOSITY_ALL] Flow 'lakehouse_dev.cloudwatch_metrics_pull.user_silver_sdp' failed to meet the expectation. Violated expectations: 'valid_activity_sequence'. Input data: '{""lakehouse_dev.cloudwatch_metrics_pull.user_bronze_sdp"":{""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""creation_date"":""03-02-2026 15:23:00"",""email"":""cherylhernandez@example.net"",""firstname"":""Peggy"",""id"":681,""last_activity_date"":""01-16-2026 22:57:39"",""last_ip"":""149.187.229.114"",""lastname"":""Shaw"",""postcode"":""50325"",""_rescued_data"":null}}'. Output record: '{""id"":681,""email"":""5fffed4717c1e398e3f0ee3da4b8ee8dc167088b"",""creation_date"":""2026-03-02T15:23:00.000Z"",""last_activity_date"":""2026-01-16T22:57:39.000Z"",""firstname"":""Peggy"",""lastname"":""Shaw"",""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""last_ip"":""149.187.229.114"",""postcode"":""50325""}'. Missing input data: false",valid_activity_sequence,"{""lakehouse_dev.cloudwatch_metrics_pull.user_bronze_sdp"":{""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""creation_date"":""03-02-2026 15:23:00"",""email"":""cherylhernandez@example.net"",""firstname"":""Peggy"",""id"":681,""last_activity_date"":""01-16-2026 22:57:39"",""last_ip"":""149.187.229.114"",""lastname"":""Shaw"",""postcode"":""50325"",""_rescued_data"":null}}'. Output record: '{""id"":681,""email"":""5fffed4717c1e398e3f0ee3da4b8ee8dc167088b"",""creation_date"":""2026-03-02T15:23:00.000Z"",""last_activity_date"":""2026-01-16T22:57:39.000Z"",""firstname"":""Peggy"",""lastname"":""Shaw"",""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""last_ip"":""149.187.229.114"",""postcode"":""50325""}",87915828-1d04-4162-b111-35df93ee5e15
2026-03-24T10:35:48.787Z,"[EXPECTATION_VIOLATION.VERBOSITY_ALL] Flow 'lakehouse_dev.cloudwatch_metrics_pull.user_silver_sdp' failed to meet the expectation. Violated expectations: 'valid_activity_sequence'. Input data: '{""lakehouse_dev.cloudwatch_metrics_pull.user_bronze_sdp"":{""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""creation_date"":""03-02-2026 15:23:00"",""email"":""cherylhernandez@example.net"",""firstname"":""Peggy"",""id"":681,""last_activity_date"":""01-16-2026 22:57:39"",""last_ip"":""149.187.229.114"",""lastname"":""Shaw"",""postcode"":""50325"",""_rescued_data"":null}}'. Output record: '{""id"":681,""email"":""5fffed4717c1e398e3f0ee3da4b8ee8dc167088b"",""creation_date"":""2026-03-02T15:23:00.000Z"",""last_activity_date"":""2026-01-16T22:57:39.000Z"",""firstname"":""Peggy"",""lastname"":""Shaw"",""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""last_ip"":""149.187.229.114"",""postcode"":""50325""}'. Missing input data: false",valid_activity_sequence,"{""lakehouse_dev.cloudwatch_metrics_pull.user_bronze_sdp"":{""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""creation_date"":""03-02-2026 15:23:00"",""email"":""cherylhernandez@example.net"",""firstname"":""Peggy"",""id"":681,""last_activity_date"":""01-16-2026 22:57:39"",""last_ip"":""149.187.229.114"",""lastname"":""Shaw"",""postcode"":""50325"",""_rescued_data"":null}}'. Output record: '{""id"":681,""email"":""5fffed4717c1e398e3f0ee3da4b8ee8dc167088b"",""creation_date"":""2026-03-02T15:23:00.000Z"",""last_activity_date"":""2026-01-16T22:57:39.000Z"",""firstname"":""Peggy"",""lastname"":""Shaw"",""address"":""0468 Laura Locks, Gallegosport, NH 19306"",""city"":""New Kimberly"",""last_ip"":""149.187.229.114"",""postcode"":""50325""}",87915828-1d04-4162-b111-35df93ee5e15
2026-03-24T10:32:46.390Z,"[EXPECTATION_VIOLATION.VERBOSITY_ALL] Flow 'lakehouse_dev.cloudwatch_metrics_pull.user_silver_sdp' failed to meet the expectation. Violated expectations: 'valid_act

## ALL WITHIN 20 MIN WINDOW

In [0]:
%sql
SELECT
  timestamp,
  -- Extracting the specific rule name
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,
  -- Extracting the record snippet for the CloudWatch log
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,
  origin.pipeline_id,
  origin.update_id
FROM event_log("87915828-1d04-4162-b111-35df93ee5e15")
WHERE level = 'ERROR'
  AND timestamp >= (current_timestamp() - INTERVAL 20 MINUTES)
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

timestamp,violated_rule,offending_record,pipeline_id,update_id


### Adding the new rule to expectations

In [0]:
%sql
-- INSERT INTO lakehouse_dev.cloudwatch_metrics_pull.expectations (tag, name, constraint)
-- VALUES (
--   'user_silver_sdp',
--   'valid_activity_sequence',
--   'last_activity_date >= creation_date'
-- );

## Data Verification

In [0]:
spark.read.csv("/Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/spend_csv/*.csv", header=True, inferSchema=True).display()

id,age,annual_income,spending_score
855,52,469.4,44.7
152,55,767.5,16.6
784,79,994.8,74.7
403,57,84.5,61.5
637,64,819.5,70.9


In [0]:
spark.read.json("/Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/dev/users_json//*.json").display()

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode
"5085 Mark Extension, Lake Dianashire, LA 57861",East David,02-06-2026 22:30:53,ismith@example.org,Heather,855,03-13-2026 23:37:39,76.240.37.16,Gregory,33847
"64050 Potter Trafficway, South Michaelport, GA 62507",West Dave,01-04-2026 05:07:35,leejennifer@example.com,Mariah,152,03-14-2026 09:58:54,160.241.38.88,Miller,94878
"6306 Rocha Rapid, Catherinemouth, CT 38427",Markhaven,01-25-2026 12:30:37,huntermorales@example.com,Sarah,784,03-19-2026 13:22:51,220.248.12.12,Gordon,54445
"10592 Figueroa Heights, Port David, AZ 72753",Port Alejandroshire,02-12-2026 22:20:25,mary02@example.com,Claire,403,03-12-2026 04:44:08,78.5.177.135,Porter,33673
"99398 Key Lake, Susanside, WV 92934",North Jean,02-28-2026 01:16:23,stephaniewhitehead@example.net,Christopher,637,03-13-2026 10:33:27,120.227.72.15,Rodriguez,73320
"02427 Dennis Corners Suite 196, Millerstad, GA 89002",Brettshire,02-19-2026 15:39:36,jackfritz@example.com,Taylor,null,03-22-2026 02:48:26,160.158.185.204,Ortiz,95965


In [0]:
%sql
select * from lakehouse_dev.cloudwatch_metrics_pull.expectations

tag,name,constraint
user_bronze_sdp,correct_schema,_rescued_data IS NULL
user_silver_sdp,valid_id,id IS NOT NULL AND id > 0
user_silver_sdp,valid_activity_sequence,last_activity_date >= creation_date
spend_silver_sdp,valid_id,id IS NOT NULL AND id > 0
user_gold_sdp,valid_age,age IS NOT NULL
user_gold_sdp,valid_income,annual_income IS NOT NULL
user_gold_sdp,valid_score,spending_score IS NOT NULL


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.raw_user_data")
display(df)

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data
"5085 Mark Extension, Lake Dianashire, LA 57861",East David,02-06-2026 22:30:53,ismith@example.org,Heather,855,03-13-2026 23:37:39,76.240.37.16,Gregory,33847,null
"64050 Potter Trafficway, South Michaelport, GA 62507",West Dave,01-04-2026 05:07:35,leejennifer@example.com,Mariah,152,03-14-2026 09:58:54,160.241.38.88,Miller,94878,null
"6306 Rocha Rapid, Catherinemouth, CT 38427",Markhaven,01-25-2026 12:30:37,huntermorales@example.com,Sarah,784,03-19-2026 13:22:51,220.248.12.12,Gordon,54445,null
"10592 Figueroa Heights, Port David, AZ 72753",Port Alejandroshire,02-12-2026 22:20:25,mary02@example.com,Claire,403,03-12-2026 04:44:08,78.5.177.135,Porter,33673,null
"99398 Key Lake, Susanside, WV 92934",North Jean,02-28-2026 01:16:23,stephaniewhitehead@example.net,Christopher,637,03-13-2026 10:33:27,120.227.72.15,Rodriguez,73320,null
"02427 Dennis Corners Suite 196, Millerstad, GA 89002",Brettshire,02-19-2026 15:39:36,jackfritz@example.com,Taylor,null,03-22-2026 02:48:26,160.158.185.204,Ortiz,95965,null


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.raw_spend_data")
display(df)

id,age,annual_income,spending_score,_rescued_data
855,52,469.4,44.7,null
152,55,767.5,16.6,null
784,79,994.8,74.7,null
403,57,84.5,61.5,null
637,64,819.5,70.9,null
929,62,505.9,33.8,null


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_bronze_sdp").filter("id is NULL")
display(df)

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_silver_sdp")
display(df)

id,email,creation_date,last_activity_date,firstname,lastname,address,city,last_ip,postcode
855,757111ddd6cff48bbfaf23081126dadc03a820a6,2026-02-06T22:30:53.000Z,2026-03-13T23:37:39.000Z,Heather,Gregory,"5085 Mark Extension, Lake Dianashire, LA 57861",East David,76.240.37.16,33847
152,0b361af27c1e6600f2658a4d401c64a29e9f042a,2026-01-04T05:07:35.000Z,2026-03-14T09:58:54.000Z,Mariah,Miller,"64050 Potter Trafficway, South Michaelport, GA 62507",West Dave,160.241.38.88,94878
784,f29a9d70b87e4d7966408248dd5a23023aa17c58,2026-01-25T12:30:37.000Z,2026-03-19T13:22:51.000Z,Sarah,Gordon,"6306 Rocha Rapid, Catherinemouth, CT 38427",Markhaven,220.248.12.12,54445
403,0870d4821a8bb73a2bdb0cc556bf6db135aeb643,2026-02-12T22:20:25.000Z,2026-03-12T04:44:08.000Z,Claire,Porter,"10592 Figueroa Heights, Port David, AZ 72753",Port Alejandroshire,78.5.177.135,33673
637,a79db1ec4799c24e57f05f2bf782ab6e7e650713,2026-02-28T01:16:23.000Z,2026-03-13T10:33:27.000Z,Christopher,Rodriguez,"99398 Key Lake, Susanside, WV 92934",North Jean,120.227.72.15,73320


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_silver_sdp")
display(df)

id,email,creation_date,last_activity_date,firstname,lastname,address,city,last_ip,postcode
855,757111ddd6cff48bbfaf23081126dadc03a820a6,2026-02-06T22:30:53.000Z,2026-03-13T23:37:39.000Z,Heather,Gregory,"5085 Mark Extension, Lake Dianashire, LA 57861",East David,76.240.37.16,33847
152,0b361af27c1e6600f2658a4d401c64a29e9f042a,2026-01-04T05:07:35.000Z,2026-03-14T09:58:54.000Z,Mariah,Miller,"64050 Potter Trafficway, South Michaelport, GA 62507",West Dave,160.241.38.88,94878
784,f29a9d70b87e4d7966408248dd5a23023aa17c58,2026-01-25T12:30:37.000Z,2026-03-19T13:22:51.000Z,Sarah,Gordon,"6306 Rocha Rapid, Catherinemouth, CT 38427",Markhaven,220.248.12.12,54445
403,0870d4821a8bb73a2bdb0cc556bf6db135aeb643,2026-02-12T22:20:25.000Z,2026-03-12T04:44:08.000Z,Claire,Porter,"10592 Figueroa Heights, Port David, AZ 72753",Port Alejandroshire,78.5.177.135,33673
637,a79db1ec4799c24e57f05f2bf782ab6e7e650713,2026-02-28T01:16:23.000Z,2026-03-13T10:33:27.000Z,Christopher,Rodriguez,"99398 Key Lake, Susanside, WV 92934",North Jean,120.227.72.15,73320


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.spend_silver_sdp")
display(df)

id,age,annual_income,spending_score,_rescued_data,spending_efficiency
855,52,469.4,44.7,null,9.522795343878302
152,55,767.5,16.6,null,2.1628664992142967
784,79,994.8,74.7,null,7.509046830003406
403,57,84.5,61.5,null,72.7810650887574
637,64,819.5,70.9,null,8.65161702573263


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_quarantine_sdp")
display(df)

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data,quarantine_reason,quarantine_ts


## Expectations

In [0]:
%sql
-- drop table lakehouse_dev.cloudwatch_metrics_pull.expectations

In [0]:

data = [
 # tag/table name      name              constraint
 ("user_bronze_sdp",  "correct_schema", "_rescued_data IS NULL"),
 ("user_silver_sdp",  "valid_id",       "id IS NOT NULL AND id > 0"),
 ("user_silver_sdp",  "valid_activity_sequence", "last_activity_date >= creation_date"),
 ("spend_silver_sdp", "valid_id",       "id IS NOT NULL AND id > 0"),
 ("user_gold_sdp",    "valid_age",      "age IS NOT NULL"),
 ("user_gold_sdp",    "valid_income",   "annual_income IS NOT NULL"),
 ("user_gold_sdp",    "valid_score",    "spending_score IS NOT NULL")

]
# spark.createDataFrame(data=data, schema=["tag", "name", "constraint"]).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.expectations")